# Malware hash threat hunt

Join **PE import features** with **MalwareBazaar threat intelligence** on **MD5 hash**, then chart families and suspicious Windows API import flags.

**Data plane:** CSV retrieval and the hunt **join** run in **Cribl Search** (Generic HTTP dataset provider + `$vt_lookups`, KQL) — not via Pyodide `fetch` and **not** through `config/proxies.yml`.

**Run All** works out of the box: no Auth-Key, no hosted PE file. Open this copy from **Welcome → Examples** (not an old saved tab).

## What you will do

| Step | Question it answers |
|------|---------------------|
| Load TI | What hashes did MalwareBazaar see recently? |
| Build PE lookup | Which TI hashes get demo import-feature flags? |
| Publish TI lookup | What labels (`signature`, type, first seen) attach to each MD5? |
| Hunt join | Which PE profiles share an MD5 with TI? |
| Charts | Which families dominate hits; which imports look hot? |

```
MalwareBazaar CSV  ──HTTP dataset──►  mb_ti  ──┬──► PE lookup (demo flags)
 (public URL)                                   └──► TI lookup (labels)
                                                      │
                                                      ▼
                                            KQL inner join on MD5
```

**TI source (default):** [MalwareBazaar public `recent` CSV](https://bazaar.abuse.ch/export/csv/recent/) — no Auth-Key. **PE source (default):** teaching import flags from TI hashes (prefers **exe/dll** when present; otherwise all valid MD5 rows — the public `recent` feed is often mostly ELF). **Optional:** authenticated MalwareBazaar export or real [IEEE/Kaggle PE imports](https://ieee-dataport.org/open-access/malware-analysis-datasets-top-1000-pe-imports) via the **Annex**.

**Related:** `Threat_Hunting_Playbook.ipynb`, `Cribl_Search_Lookup_Magics.ipynb`, `Cribl_API_Examples.ipynb`.

## Prerequisites

1. Hosted Cribl app with **Cribl Search**, `%%cribl_search`, and lookup magics.
2. **No MalwareBazaar Auth-Key** on the default path — TI uses `https://bazaar.abuse.ch/export/csv/recent/`.
3. **Optional:** `MALWAREBAZAAR_AUTH_KEY` (authenticated export) or `MALWARE_HUNT_PE_CSV_URL` + **Annex** for real IEEE/Kaggle PE CSV.
4. **No `proxies.yml` changes** — Search workers fetch the public MalwareBazaar URL. If Search only returns the CSV `#` preamble, the notebook uses a **bundled teaching TI subset** (80 exe rows) so Run All still completes.
5. Lookups ≤ **10,000** rows. Heavy Search cells use `timeout=300` (seconds).

In [ ]:
# TI: public MalwareBazaar recent CSV (no Auth-Key). Optional authenticated export or IEEE PE CSV.
import os

MB_PUBLIC_RECENT = "https://bazaar.abuse.ch/export/csv/recent/"
MB_EXPORT_BASE = "https://mb-api.abuse.ch/v2/files/exports"

TI_CSV_URL = MB_PUBLIC_RECENT
_mb_auth = (os.environ.get("MALWAREBAZAAR_AUTH_KEY") or "").strip()
if _mb_auth:
    TI_CSV_URL = f"{MB_EXPORT_BASE}/{_mb_auth}/recent.csv"

PE_CSV_URL = (os.environ.get("MALWARE_HUNT_PE_CSV_URL") or "").strip()
USE_IEEE_PE_HTTP = PE_CSV_URL.startswith("https://")

print("TI (Search HTTP GET):", TI_CSV_URL)
print("PE HTTP (optional Annex):", PE_CSV_URL or "(demo flags from TI in §C)")
print("USE_IEEE_PE_HTTP:", USE_IEEE_PE_HTTP)

In [ ]:
import io

import pandas as pd

HUNT_IMPORT_COLUMNS = [
    "GetProcAddress",
    "LoadLibraryA",
    "VirtualAlloc",
    "VirtualProtect",
    "WriteProcessMemory",
    "CreateRemoteThread",
    "WinExec",
    "ShellExecuteA",
    "URLDownloadToFileA",
    "InternetOpenA",
    "RegSetValueExA",
    "CreateProcessA",
    "ExitProcess",
    "IsDebuggerPresent",
    "NtQueryInformationProcess",
]


def _pick_col(frame: pd.DataFrame, *names: str) -> str | None:
    keys = {str(c).strip().lower(): c for c in frame.columns}
    for n in names:
        if n in keys:
            return keys[n]
    return None


MB_CSV_HEADER = (
    '"first_seen_utc","sha256_hash","md5_hash","sha1_hash","reporter","file_name",'
    '"file_type_guess","mime_type","signature","clamav","vtpercent","imphash","ssdeep","tlsh"'
)


def _mb_csv_lines_from_search(frame: pd.DataFrame) -> list[str]:
    """Collect MalwareBazaar CSV lines from Search HTTP / externaldata (_raw / Event)."""
    raw_col = _pick_col(frame, "_raw", "event")
    if raw_col is None:
        return []
    lines: list[str] = []
    for blob in frame[raw_col].astype(str):
        lines.extend(blob.splitlines())
    if len(lines) < 15 and len(frame) == 1 and raw_col is not None:
        blob = str(frame[raw_col].iloc[0])
        if len(blob) > 400:
            lines = blob.splitlines()
    return lines


def _looks_like_md5_series(series: pd.Series) -> bool:
    sample = series.dropna().astype(str).str.strip().str.lower().head(24)
    if sample.empty:
        return False
    return float(sample.str.fullmatch(r"[0-9a-f]{32}").mean()) > 0.5


def _parse_mb_csv_lines(lines: list[str]) -> pd.DataFrame:
    header_line = None
    data_lines: list[str] = []
    for ln in lines:
        s = ln.strip()
        if not s:
            continue
        if "md5_hash" in s.lower():
            header_line = s.lstrip("#").strip()
            continue
        if s.startswith("#"):
            continue
        data_lines.append(s)
    if not header_line:
        header_line = MB_CSV_HEADER
    if not data_lines:
        raise ValueError(
            "MalwareBazaar CSV missing data rows (only comment preamble?). "
            f"Got {len(lines)} line(s) from Search; re-run §B provider + dataset + search."
        )
    return pd.read_csv(io.StringIO(header_line + "\n" + "\n".join(data_lines)))


def _normalize_mb_ti_frame(frame: pd.DataFrame) -> pd.DataFrame:
    """MalwareBazaar table from typed CSV columns or line-per-event / full-body `_raw`."""
    md5_col = _pick_col(frame, "md5", "md5_hash", "hash")
    if md5_col and _looks_like_md5_series(frame[md5_col]):
        out = frame.copy()
    else:
        lines = _mb_csv_lines_from_search(frame)
        if not lines:
            raise ValueError(
                "Expected MalwareBazaar columns or CSV lines in _raw/Event. "
                f"Got columns: {list(frame.columns)}"
            )
        out = _parse_mb_csv_lines(lines)
    out.columns = [str(c).strip().strip('"') for c in out.columns]
    for col in out.columns:
        if out[col].dtype == object:
            out[col] = out[col].astype(str).str.strip().str.strip('"')
    return out


TI_FALLBACK_CSV = """
md5,signature,tags,malware_family,file_type,first_seen
d24dd5333f66d721093104cc09899107,Win32.Emotet,"trojan,dropper",emotet,exe,2024-01-15
35a245209e884d51f187ceb75429f0cf,Win32.Qakbot,"banker,trojan",qakbot,exe,2024-02-15
fc325f0bac11155ad3de42fdbed08e9d,Win32.BazarLoader,"loader,trojan",bazarloader,exe,2024-03-15
73214bbc438f3c271ac3f62bc08088a7,MSIL.AgentTesla,"stealer,trojan",agenttesla,exe,2024-04-15
78703cb836d57010c3e779d4fdb159c9,Win32.FormBook,"stealer,trojan",formbook,exe,2024-05-15
ea2132ef163d343c6546158181d5c737,Win32.Remcos,"rat,trojan",remcos,exe,2024-06-15
2912d8a7c269a34fbc6f92fe4adbf820,Win32.njRAT,"rat,trojan",njrat,exe,2024-07-15
33af35b448c0e5f00377290b877e89f0,PUA.OptionalBundler,pua,pua,exe,2024-08-15
7733b0c2d289c30c334b27347b7985d6,Win32.Emotet,"trojan,dropper",emotet,exe,2024-09-15
19b8e5bf29c88c086e6e0b851022d65d,Win32.Qakbot,"banker,trojan",qakbot,exe,2024-10-15
0b7a3db9da99c82bf60801257097407b,Win32.BazarLoader,"loader,trojan",bazarloader,exe,2024-11-15
f0fbe82b5c1ee21ffa62538b27dfc2bc,MSIL.AgentTesla,"stealer,trojan",agenttesla,exe,2024-12-15
a305aca5994a105a1c5b5edff18207ac,Win32.FormBook,"stealer,trojan",formbook,exe,2024-01-15
fb2efd7a8a8c7a7e8001b1d1067420f1,Win32.Remcos,"rat,trojan",remcos,exe,2024-02-15
c8823db912fbf5321bba9fbc27fa25bf,Win32.njRAT,"rat,trojan",njrat,exe,2024-03-15
11281e471960619b09e96f03e089fb61,PUA.OptionalBundler,pua,pua,exe,2024-04-15
a6ff770ea3cf601e97280e336e2f9d4e,Win32.Emotet,"trojan,dropper",emotet,exe,2024-05-15
0c34db57973ee4a1645c941945c77453,Win32.Qakbot,"banker,trojan",qakbot,exe,2024-06-15
ac2b1cb5f4cac2252a48b0af6d1cba9e,Win32.BazarLoader,"loader,trojan",bazarloader,exe,2024-07-15
7ffb4440db27b54e9fda6d76893cafff,MSIL.AgentTesla,"stealer,trojan",agenttesla,exe,2024-08-15
e38aa174163b6b8561a41caf9f190408,Win32.FormBook,"stealer,trojan",formbook,exe,2024-09-15
ac3dd56dae6212f1e451ad44b9af7870,Win32.Remcos,"rat,trojan",remcos,exe,2024-10-15
04707ffbf685fc1a358e0f99be4cce9a,Win32.njRAT,"rat,trojan",njrat,exe,2024-11-15
b2b8a16895092df5831a844e8996c106,PUA.OptionalBundler,pua,pua,exe,2024-12-15
77bc765c70ae7dd23b9d818fabbc0186,Win32.Emotet,"trojan,dropper",emotet,exe,2024-01-15
29514a01d3449a863e4dbb0f98d00f0f,Win32.Qakbot,"banker,trojan",qakbot,exe,2024-02-15
e46703f6c54eaad74716305933ddc2cb,Win32.BazarLoader,"loader,trojan",bazarloader,exe,2024-03-15
2ce2e7465e18642b05d09e3e68b39f25,MSIL.AgentTesla,"stealer,trojan",agenttesla,exe,2024-04-15
fbc7e5833f50254ec12f71eb3880ae14,Win32.FormBook,"stealer,trojan",formbook,exe,2024-05-15
6b81642a7b4bbdd2c80b25a1c14288ec,Win32.Remcos,"rat,trojan",remcos,exe,2024-06-15
3316eaba45d7108a98545d5636888b69,Win32.njRAT,"rat,trojan",njrat,exe,2024-07-15
286f4ec271c1335079ed74b918ba5d15,PUA.OptionalBundler,pua,pua,exe,2024-08-15
be41b5dd6924ad087c248e30bd2450f1,Win32.Emotet,"trojan,dropper",emotet,exe,2024-09-15
c6cbb3e556433ececcfa51f8890a7c58,Win32.Qakbot,"banker,trojan",qakbot,exe,2024-10-15
3a79b528bf0b663de6e41a3109db6897,Win32.BazarLoader,"loader,trojan",bazarloader,exe,2024-11-15
1c4b77c5404d672e6d6c4d62e36a00d0,MSIL.AgentTesla,"stealer,trojan",agenttesla,exe,2024-12-15
fcfb88f9e75b73594567829e0d5d4469,Win32.FormBook,"stealer,trojan",formbook,exe,2024-01-15
25bbbe8961a3098f4ec80d806e18f221,Win32.Remcos,"rat,trojan",remcos,exe,2024-02-15
6411c1567e57678992754ca04c952b60,Win32.njRAT,"rat,trojan",njrat,exe,2024-03-15
8a6dbce8f01a34c6b296c8383b787f4e,PUA.OptionalBundler,pua,pua,exe,2024-04-15
ab2eb23594f9f31618a84c2dcf89e0c4,Win32.Emotet,"trojan,dropper",emotet,exe,2024-05-15
c9ca4ea4cace48d407935a5c617e89d1,Win32.Qakbot,"banker,trojan",qakbot,exe,2024-06-15
84555491dac55b5fea6b8dd01ff5b9cf,Win32.BazarLoader,"loader,trojan",bazarloader,exe,2024-07-15
1731fa5836e38797d1948a77cfa178d3,MSIL.AgentTesla,"stealer,trojan",agenttesla,exe,2024-08-15
1388f8a9939b3fbfef993b7caf8cd2de,Win32.FormBook,"stealer,trojan",formbook,exe,2024-09-15
dacd5e551ec83505bef3fec6c871b61c,Win32.Remcos,"rat,trojan",remcos,exe,2024-10-15
d6fdbb033d5147bfcb5c75fbdb9c1e62,Win32.njRAT,"rat,trojan",njrat,exe,2024-11-15
00b86c8308aa0bf668dc94fb2d4a9573,PUA.OptionalBundler,pua,pua,exe,2024-12-15
10df642e3906d54b57311ff6814ebf4f,Win32.Emotet,"trojan,dropper",emotet,exe,2024-01-15
28d006d728c251252bf9c33fcaed3831,Win32.Qakbot,"banker,trojan",qakbot,exe,2024-02-15
a339734e41e6525dd4f7138f948f4f27,Win32.BazarLoader,"loader,trojan",bazarloader,exe,2024-03-15
6ebdfefbd11cc8fd5ca21e6e40440af2,MSIL.AgentTesla,"stealer,trojan",agenttesla,exe,2024-04-15
4de69e94f781df03a52ff5924762299c,Win32.FormBook,"stealer,trojan",formbook,exe,2024-05-15
44389abb6a4dc17def5eaa3009f4c36f,Win32.Remcos,"rat,trojan",remcos,exe,2024-06-15
5093f2e37954944bcf4fc23e4b11bb14,Win32.njRAT,"rat,trojan",njrat,exe,2024-07-15
51acc2c8230b79f8ee16b71384237cc2,PUA.OptionalBundler,pua,pua,exe,2024-08-15
7fa59b94f6d8bf2198f186f0c5301a59,Win32.Emotet,"trojan,dropper",emotet,exe,2024-09-15
d22ec1cd3dea597415f4b179635c0556,Win32.Qakbot,"banker,trojan",qakbot,exe,2024-10-15
c0a301cb6f68ae96d9cbaf3ac497a834,Win32.BazarLoader,"loader,trojan",bazarloader,exe,2024-11-15
a235fd8856f2f524438c804c6da612ab,MSIL.AgentTesla,"stealer,trojan",agenttesla,exe,2024-12-15
a69f7ecc04d226bde17c2684aeb01759,Win32.FormBook,"stealer,trojan",formbook,exe,2024-01-15
d3be916737963b4babdef1337469d03c,Win32.Remcos,"rat,trojan",remcos,exe,2024-02-15
687bd70b60c9740e10974e46c9e8d307,Win32.njRAT,"rat,trojan",njrat,exe,2024-03-15
2a70601eb7c7be3837a3a7712d1a9eb8,PUA.OptionalBundler,pua,pua,exe,2024-04-15
0f2f1a0ee2d1a305ae81831301c0c010,Win32.Emotet,"trojan,dropper",emotet,exe,2024-05-15
d37328ca5add8ca336d26c8d5ef484aa,Win32.Qakbot,"banker,trojan",qakbot,exe,2024-06-15
cf8affb65f11541646f01599b7cecf9b,Win32.BazarLoader,"loader,trojan",bazarloader,exe,2024-07-15
15fb51351dadca50d84ee2f4a086bf2a,MSIL.AgentTesla,"stealer,trojan",agenttesla,exe,2024-08-15
a4fa3be03c1d7a8124be54f29462ba5b,Win32.FormBook,"stealer,trojan",formbook,exe,2024-09-15
017d12d92bfec16e36e904e4f6ba2f72,Win32.Remcos,"rat,trojan",remcos,exe,2024-10-15
57d928aace6fcc43a2f6768825d0d2da,Win32.njRAT,"rat,trojan",njrat,exe,2024-11-15
8b7ca73c638a7844a637b15aac32900c,PUA.OptionalBundler,pua,pua,exe,2024-12-15
b52afbab826d50b10a27741f943a1cec,Win32.Emotet,"trojan,dropper",emotet,exe,2024-01-15
d6baebb7701e8dfddf294b3ece3779b9,Win32.Qakbot,"banker,trojan",qakbot,exe,2024-02-15
e77874c2b5dccd718ceebb80a03bf0d0,Win32.BazarLoader,"loader,trojan",bazarloader,exe,2024-03-15
f2ed0b4a6429537da79dec22e8379f91,MSIL.AgentTesla,"stealer,trojan",agenttesla,exe,2024-04-15
37d849c2342f4c2dbaccd8f079d9ec41,Win32.FormBook,"stealer,trojan",formbook,exe,2024-05-15
1fd334b90026a1e7dab048f4c48c40e9,Win32.Remcos,"rat,trojan",remcos,exe,2024-06-15
35388dd59522f1846fd9894bff5b0d66,Win32.njRAT,"rat,trojan",njrat,exe,2024-07-15
992c178c0e0a50ff0222f6d231024b41,PUA.OptionalBundler,pua,pua,exe,2024-08-15
"""


def _load_mb_ti_fallback_static() -> pd.DataFrame:
    """Teaching TI when Search only ingests the MalwareBazaar # comment preamble."""
    out = pd.read_csv(io.StringIO(TI_FALLBACK_CSV))
    out.columns = [str(c).strip() for c in out.columns]
    return out


def load_mb_ti_from_search(frame: pd.DataFrame) -> pd.DataFrame:
    try:
        out = _normalize_mb_ti_frame(frame)
        if len(out) >= 20:
            return out
        print(f"Search TI only {len(out)} row(s); using bundled teaching fallback.")
    except ValueError as err:
        print(f"Search TI parse: {err}")
    print(
        "Using bundled teaching TI (80 rows, exe samples) — live MalwareBazaar CSV had only comment lines in Search."
    )
    return _load_mb_ti_fallback_static()


print("Helpers loaded (import columns + MalwareBazaar CSV normalizer).")

## A0) Reset lookups (idempotent)

Clears prior hunt artifacts so **Run All** is repeatable.


In [ ]:
%%cribl_delete_search_lookup notebook_pe_features.csv

In [ ]:
%%cribl_delete_search_lookup notebook_mb_ti.csv

In [ ]:
%%cribl_api DELETE /m/default_search/search/datasets/notebook_ti_imports var=_ti_ds_a0 ignoreFailure=true response=json

In [ ]:
%%cribl_api DELETE /m/default_search/search/dataset-providers/notebook_ti_http var=_ti_prov_a0 ignoreFailure=true response=json

## B) Load MalwareBazaar TI (HTTP dataset provider)

The public `recent` CSV has a `#` comment preamble. On many tenants, `externaldata` with `CSV Datatypes` only materializes those comment lines — not the data rows. We register a **Generic HTTP API** provider + federated dataset (same URL, Search-side GET), then query it and normalize in Python (structured columns, line-per-event `_raw`, or one full-body `_raw`).


In [ ]:
%%cribl_api POST /m/default_search/search/dataset-providers var=ti_provider_res response=json template=on
json:
  type: api_http
  id: notebook_ti_http
  description: MalwareBazaar recent CSV (malware hash hunt)
  authenticationMethod: none
  availableEndpoints:
    - name: ti_recent
      method: GET
      url: "{{ TI_CSV_URL }}"
      headers: []
      dataField: ""

In [ ]:
%%cribl_api POST /m/default_search/search/datasets var=ti_dataset_res response=json
json:
  type: api_http
  id: notebook_ti_imports
  description: MalwareBazaar TI for hash threat hunt
  provider: notebook_ti_http
  enabledEndpoints:
    - ti_recent
  filter: "true"
  searchVersion: v1
  breakerRulesets:
    - Cribl Search
  metadata:
    enableAcceleration: false

In [ ]:
%%cribl_search var=mb_ti_raw lang=kql limit=5000 preview=true earliest=-1h latest=now timeout=300
dataset="notebook_ti_imports"
| limit 5000

In [ ]:
mb_ti = load_mb_ti_from_search(mb_ti_raw).head(5000)

md5_col = _pick_col(mb_ti, "md5", "md5_hash", "hash")
sig_col = _pick_col(mb_ti, "signature")
ftype_col = _pick_col(mb_ti, "file_type", "file_type_guess")
exe_n = 0
if ftype_col:
    exe_n = int(mb_ti[ftype_col].astype(str).str.lower().isin(["exe", "dll"]).sum())

print(f"TI rows: {len(mb_ti):,}  |  columns: {list(mb_ti.columns)[:6]}…")
print(f"MD5 column: {md5_col!r}  |  signatures: {sig_col!r}  |  exe/dll rows: {exe_n:,}")
mb_ti.head(5)

## C) Build demo PE import lookup (exe/dll)

MalwareBazaar `recent` has hashes and labels but not PE import flags. We derive **teaching** import flags from TI MD5s (prefer **exe/dll**; fall back to any hash when the feed has none) so the join mechanics match production (real `top_1000_pe_imports.csv` → **Annex**).


In [ ]:
md5_col = _pick_col(mb_ti, "md5", "md5_hash", "hash")
ftype_col = _pick_col(mb_ti, "file_type", "file_type_guess")
if md5_col is None:
    raise ValueError(f"Expected md5 column; got {list(mb_ti.columns)}")

pe_src = mb_ti.copy()
if ftype_col:
    pe_exe = pe_src[pe_src[ftype_col].astype(str).str.lower().isin(["exe", "dll"])]
    if len(pe_exe) > 0:
        pe_src = pe_exe
        print(f"PE demo source: {len(pe_src):,} exe/dll rows")
    else:
        print("PE demo source: no exe/dll in recent feed — using all valid MD5 rows")

pe_lookup_df = pd.DataFrame()
pe_lookup_df["hash"] = pe_src[md5_col].astype(str).str.strip().str.lower()
pe_lookup_df = pe_lookup_df.dropna(subset=["hash"])
pe_lookup_df = pe_lookup_df[pe_lookup_df["hash"].str.len() == 32]
pe_lookup_df = pe_lookup_df.drop_duplicates(subset=["hash"]).head(2000)

for i, col in enumerate(HUNT_IMPORT_COLUMNS):
    pe_lookup_df[col] = [
        1 if (int(h[:8], 16) + len(col) + j) % 4 < 2 else 0 for j, h in enumerate(pe_lookup_df["hash"])
    ]

print("PE demo lookup rows:", len(pe_lookup_df))
pe_lookup_df.head(8)

In [ ]:
%%cribl_save_search_lookup notebook_pe_features.csv var=pe_lookup_df replace=true

## E) Normalize MD5 and publish TI lookup

In [ ]:
md5_col = _pick_col(mb_ti, "md5", "md5_hash", "hash")
if md5_col is None:
    raise ValueError(f"Expected md5 column; got {list(mb_ti.columns)}")

mb_lookup_df = pd.DataFrame()
mb_lookup_df["md5"] = mb_ti[md5_col].astype(str).str.strip().str.lower()
for col in (
    "signature",
    "tags",
    "malware_family",
    "file_type",
    "file_type_guess",
    "first_seen",
    "first_seen_utc",
):
    src = _pick_col(mb_ti, col)
    if src:
        mb_lookup_df[col] = mb_ti[src]

mb_lookup_df = mb_lookup_df.dropna(subset=["md5"])
mb_lookup_df = mb_lookup_df[mb_lookup_df["md5"].str.len() == 32]
mb_lookup_df = mb_lookup_df.drop_duplicates(subset=["md5"]).head(10000)

if "malware_family" not in mb_lookup_df.columns and "signature" in mb_lookup_df.columns:
    mb_lookup_df["malware_family"] = mb_lookup_df["signature"].astype(str)

print("lookup rows:", len(mb_lookup_df))
mb_lookup_df.head(8)

In [ ]:
%%cribl_save_search_lookup notebook_mb_ti.csv var=mb_lookup_df replace=true

In [ ]:
%%cribl_search var=lookup_preview lang=kql limit=10 preview=true earliest=-1h latest=now
dataset="$vt_lookups" lookupFile="notebook_mb_ti"
| limit 10

## F) Hunt — join PE lookup to TI lookup on MD5

Do not start with `let` — `%%cribl_search` prepends `cribl` only when the body begins with `dataset=`.

In [ ]:
%%cribl_search var=hunt_hits lang=kql limit=5000 preview=true earliest=-1h latest=now timeout=300
dataset="$vt_lookups" lookupFile="notebook_pe_features"
| extend hash = tolower(tostring(hash))
| join kind=inner (
    dataset="$vt_lookups" lookupFile="notebook_mb_ti"
  ) on $left.hash == $right.md5
| project hash, malware_family, signature, tags, GetProcAddress, VirtualAlloc, WriteProcessMemory, CreateRemoteThread, URLDownloadToFileA
| limit 2000

## G) Checkpoint — match rate

In [ ]:
%%cribl_search var=pe_total lang=kql limit=0 preview=false earliest=-1h latest=now timeout=300
dataset="$vt_lookups" lookupFile="notebook_pe_features"
| summarize pe_samples = dcount(hash)

In [ ]:
import pandas as pd

hits = len(hunt_hits)
pe_n = int(pe_total.iloc[0]["pe_samples"]) if len(pe_total) else 0
rate = (hits / pe_n * 100) if pe_n else 0
print(f"Join hits: {hits} / distinct PE hashes: {pe_n} ({rate:.1f}%)")
if hits == 0:
    raise ValueError(
        "No join hits — check §B loaded TI, §C published PE lookup, and re-run A0 then Run All."
    )
hunt_hits.head(10)

## H) Visualize TI families and suspicious imports

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

SUSPICIOUS = [
    "VirtualAlloc",
    "WriteProcessMemory",
    "CreateRemoteThread",
    "URLDownloadToFileA",
    "WinExec",
    "IsDebuggerPresent",
]

fam_col = next(
    (c for c in hunt_hits.columns if str(c).lower() in ("malware_family", "signature")),
    None,
)
if fam_col is None:
    raise ValueError(f"Expected malware_family or signature in hunt_hits; got {list(hunt_hits.columns)}")

by_family = hunt_hits[fam_col].value_counts().head(8)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
by_family.sort_values().plot(kind="barh", ax=axes[0], color="#c44e52")
axes[0].set_title("Hunt hits by malware family")
axes[0].set_xlabel("Samples")

present = [c for c in SUSPICIOUS if c in hunt_hits.columns]
if present:
    means = hunt_hits[present].apply(pd.to_numeric, errors="coerce").fillna(0).mean().sort_values(ascending=False)
    means.plot(kind="bar", ax=axes[1], color="#4c72b0")
    axes[1].set_title("Mean import flag rate (matched samples)")
    axes[1].set_ylabel("Fraction with flag = 1")
    axes[1].tick_params(axis="x", rotation=45)
else:
    axes[1].text(0.5, 0.5, "Import columns not in projection", ha="center")

plt.tight_layout()
plt.show()
print("Interpretation: inner join surfaces PE rows with both import features and TI labels on the same MD5.")

## Interpretation

- **Inner join** answers: *which PE import profiles in our sample also have MalwareBazaar-style TI on the same hash?*
- **Match rate** on the demo path is often ~100% (PE hashes are a subset of the same TI feed). With real IEEE PE imports + live MalwareBazaar, overlap is usually lower.
- **MD5 reuse** — one hash can map to multiple submissions over time; treat TI as contextual, not ground truth.
- **Production** — use MalwareBazaar public or authenticated exports for TI; load real PE features from IEEE/Kaggle or your feature store (Annex / `MALWARE_HUNT_PE_CSV_URL`).

## Cleanup

In [ ]:
%%cribl_delete_search_lookup notebook_mb_ti.csv

In [ ]:
%%cribl_delete_search_lookup notebook_pe_features.csv

In [ ]:
%%cribl_api DELETE /m/default_search/search/datasets/notebook_pe_imports var=_ds_del ignoreFailure=true response=json

In [ ]:
%%cribl_api DELETE /m/default_search/search/dataset-providers/notebook_pe_http var=_prov_del ignoreFailure=true response=json

In [ ]:
%%cribl_api DELETE /m/default_search/search/datasets/notebook_ti_imports var=_ti_ds_del ignoreFailure=true response=json

In [ ]:
%%cribl_api DELETE /m/default_search/search/dataset-providers/notebook_ti_http var=_ti_prov_del ignoreFailure=true response=json

## Annex — IEEE/Kaggle PE imports via HTTP provider

When `USE_IEEE_PE_HTTP` is true (`MALWARE_HUNT_PE_CSV_URL` or uncommented `PE_CSV_URL` in the URL cell), run §A0 then the cells below, replace §C with a preview of `notebook_pe_imports`, and in §F join `dataset="notebook_pe_imports"` to `notebook_mb_ti` instead of `notebook_pe_features`.


In [ ]:
if not USE_IEEE_PE_HTTP:
    raise RuntimeError(
        "Set MALWARE_HUNT_PE_CSV_URL to a public HTTPS top_1000_pe_imports.csv before running the Annex."
    )

In [ ]:
%%cribl_api POST /m/default_search/search/dataset-providers var=pe_provider_res response=json template=on
json:
  type: api_http
  id: notebook_pe_http
  description: PE import features (malware hash hunt example)
  authenticationMethod: none
  availableEndpoints:
    - name: pe_imports
      method: GET
      url: "{{ PE_CSV_URL }}"
      headers: []
      dataField: ""

In [ ]:
%%cribl_api POST /m/default_search/search/datasets var=pe_dataset_res response=json
json:
  type: api_http
  id: notebook_pe_imports
  description: PE imports sample for hash threat hunt
  provider: notebook_pe_http
  enabledEndpoints:
    - pe_imports
  filter: "true"
  searchVersion: v1
  breakerRulesets:
    - Cribl Search
  metadata:
    enableAcceleration: false

## Troubleshooting

| Symptom | Fix |
|---------|-----|
| Only `#` comment rows / no data | Expected on some tenants — parse cell uses bundled `TI_FALLBACK_CSV`. Re-run **§B** if provider/dataset steps failed. |
| `Expected md5 column` / bad columns | Re-run **§B** + parse cell — normalizer rebuilds from `_raw` when typed columns are wrong. |
| Provider/dataset 409 | Run **A0** deletes, then **§B** again. |
| No join hits | Confirm §B shows TI rows and §C published PE lookup; re-run **A0** then **Run All**. |
| Stale notebook tab | Close tab → **Welcome** → **Malware Hash Threat Hunt** → **Open example**. |